In [ ]:
import numpy as np
import sklearn as sk
import pandas as pd
import data_management as dm
from core import feature_builders as fb
import tree_models as tm
import model_wrappers as wp
from graph_helpers import plot_var_vs_loss, plot_ewma_bank

In [2]:
TICKERS = ["AAPL", "JPM", "XOM", "KO"]

WEIGHTS = {
    "AAPL": 0.25,
    "JPM": 0.25,
    "XOM": 0.25,
    "KO": 0.25,
}

df = pd.read_csv(
    "./data_general/4_ticker_data.csv",
    parse_dates=["Date"],
    index_col="Date",
)


tickers_data = df[TICKERS].pct_change().dropna()



In [ ]:
calendar_last_180_days = dm.DailyGroundTruthCalendar(
    df, TICKERS, WEIGHTS,
    train_start="2024-06-03",   # Pazartesi, ~6 ay geriye (180 takvim günü civarı)
    train_end="2024-11-29",     # Cuma, Kasım'ın son iş günü
    forecast_start="2024-12-02",  # Pazartesi
    forecast_end="2024-12-31",
    warmup_bars= 30,
)

[WARNING] forecast_end=2024-12-31 falls on Tuesday, expected Friday (using it as-is).
[WARNING] [train] 4 weekday days had no data and were skipped (likely market holidays): 2024-06-19, 2024-07-04, 2024-09-02, 2024-11-28
[INFO] train set: 126 bars, 4 weekday days skipped
[WARNING] [forecast] 1 weekday day had no data and were skipped (likely market holidays): 2024-12-25
[INFO] forecast set: 21 bars, 1 weekday day skipped


In [4]:
training_feature_long, forecast_feature_long = fb.build_long_features(calendar_last_180_days, target_scale= 10000)

[INFO] long train: 504 rows -> 504 after dropna (0 dropped, 0.0%)
[INFO] long forecast: 88 rows -> 88 after dropna (0 dropped, 0.0%)


In [5]:
VAR_CONFIDENCE = 0.95

# Both banks are SINGLE-USE: training fills them in place and marks them trained, so they cannot be given to
# another training run (create new banks per run; creating them in this same cell keeps re-running it safe).
residual_bank = dm.ResidualBank(
    mu_mode="constant",                         # "constant": mu_hat = default_mu_hat | "rolling": mean of the last rolling_mu_hat_window_size returns
    default_mu_hat=0.0,
    rolling_mu_hat_window_size=50,              # only read when mu_mode="rolling"
)

# built from the calendar: tickers and weights are read from it (training checks it is the same calendar)
ewma_bank = dm.EwmaVarianceBank(
    calendar_last_180_days,
    ewma_lam=0.94,                              # RiskMetrics decay: larger = forgets slower
    ewma_min_periods=30,                        # the first N train bars seed the covariance (EWMA burn-in)
)

model = tm.xgboost_train(calendar_last_180_days, training_feature_long, residual_bank=residual_bank, ewma_bank=ewma_bank)

[INFO] train_dates_past_burn_in: 125/125 candidate dates are past the burn-in (burn_in_bars=29, warmup_bars=30, train_data starts 2024-06-03, burn-in ends 2024-05-31, 0 date(s) dropped).


In [6]:
# fills copies of the model's two banks with the forecast window (no VaR yet, nothing about reading involved)
residualbank, ewmabank = model.calculate_multi_day_standart_residuals_from_long_format_model(
    calendar=calendar_last_180_days,
    long_forecast_feature=forecast_feature_long,
)

# reads the VaR off the filled bank, adds the realized loss, records the run under run_key.
# Everything about HOW to read is chosen here, so the same filled bank can be read again any other way.
var_df = calendar_last_180_days.var_from_residual_bank_forecast_period(
    residualbank,
    var_confidence=VAR_CONFIDENCE,                  # e.g. 0.95 -> takes the 5% lower quantile of the residuals
    multi_day_var_calculation_window_size=30,      # int: use at most the last N residuals; None: use all residuals available
    run_key="xgboost_long",                         # key in calendar.var_prediction_runs (reusing a key overwrites that run)
    multi_day_var_min_periods=30,                   # min prior residuals a day needs to get a VaR (else ValueError)
)

In [7]:
var_df

,varForecast,n_residuals_used,loss
Date,,,
2024-12-02,0.007295,30,0.003004
2024-12-03,0.00865,30,-0.000849
2024-12-04,0.006847,30,0.013437
2024-12-05,0.023083,30,-0.006114
2024-12-06,0.00832,30,0.002476
2024-12-09,0.008813,30,0.000758
2024-12-10,0.009657,30,-0.000788
2024-12-11,0.011707,30,0.003339
2024-12-12,0.009894,30,-0.004004


In [8]:
plot_var_vs_loss(var_df, residualbank, calendar=calendar_last_180_days)   # calendar verilirse train/forecast bantlari cizilir

In [9]:
# EwmaVarianceBank.portfolio_variance: her gunun portfoy varyansi (bank'in asil ciktisi, ResidualBank'a gider)
# Egitim (96 gun) + forecast (21 gun) = 117 satir; indeks = sonucun yasandigi gun
ewmabank.portfolio_variance.head()

2024-06-04    0.000090
2024-06-05    0.000048
2024-06-06    0.000044
2024-06-07    0.000057
2024-06-10    0.000059
Name: predicted_variance, dtype: float64

In [10]:
ewmabank.portfolio_variance.tail()

2024-12-24    0.000045
2024-12-26    0.000029
2024-12-27    0.000042
2024-12-30    0.000046
2024-12-31    0.000045
Name: predicted_variance, dtype: float64

In [11]:
ewmabank.cov_by_date

{Timestamp('2024-06-04 00:00:00'):           AAPL       JPM       XOM        KO
 AAPL  0.000212  0.000003  0.000053  0.000026
 JPM   0.000003  0.000163  0.000064  0.000031
 XOM   0.000053  0.000064  0.000166  0.000024
 KO    0.000026  0.000031  0.000024  0.000035,
 Timestamp('2024-06-05 00:00:00'):           AAPL       JPM       XOM        KO
 AAPL  0.000199  0.000002  0.000049  0.000026
 JPM   0.000002  0.000163  0.000072  0.000017
 XOM   0.000049  0.000072  0.000170  0.000008
 KO    0.000026  0.000017  0.000008  0.000049,
 Timestamp('2024-06-06 00:00:00'):           AAPL       JPM       XOM        KO
 AAPL  0.000191 -0.000003  0.000048  0.000024
 JPM  -0.000003  0.000159  0.000066  0.000016
 XOM   0.000048  0.000066  0.000161  0.000007
 KO    0.000024  0.000016  0.000007  0.000046,
 Timestamp('2024-06-07 00:00:00'):           AAPL       JPM       XOM        KO
 AAPL  0.000182 -0.000002  0.000042  0.000021
 JPM  -0.000002  0.000150  0.000061  0.000015
 XOM   0.000042  0.000061  0.0001

In [12]:
# kac gunun kovaryansi kayitli ve son gunun matrisi (matrisler ticker etiketli DataFrame)
print(len(ewmabank.cov_by_date), "gun")
last_date = ewmabank.portfolio_variance.index[-1]
ewmabank.cov_by_date[last_date]

146 gun


,AAPL,JPM,XOM,KO
AAPL,0.000117,0.000048,0.000004,0.000028
JPM,0.000048,0.000234,0.000044,0.000001
XOM,0.000004,0.000044,0.000095,0.000033
KO,0.000028,0.000001,0.000033,0.000071


In [13]:
plot_ewma_bank(ewmabank)                                                  # varsayılan


In [14]:
plot_ewma_bank(ewmabank, split_date=calendar_last_180_days.train_data_last_index[0])   # eğitim/forecast sınırı


In [15]:
plot_ewma_bank(ewmabank, as_volatility=True)                              # sqrt ve % olarak


In [16]:
# bank'in train gunleri icin VaR (in-sample); ilk multi_day_var_min_periods gun onceki residual olmadigi icin dusulur
train_var = calendar_last_180_days.var_from_residual_bank_train_period(
    residualbank,
    var_confidence=VAR_CONFIDENCE,
    multi_day_var_calculation_window_size=10,
    multi_day_var_min_periods=10,
)
train_var

,varForecast,n_residuals_used,loss
Date,,,
2024-06-18,0.003976,10,-0.002237
2024-06-20,0.004838,10,-0.000339
2024-06-21,0.00451,10,0.005414
2024-06-24,0.004815,10,-0.016274
2024-06-25,0.005482,10,-0.000292
...,...,...,...
2024-11-22,0.005574,10,-0.005688
2024-11-25,0.005251,10,-0.003076
2024-11-26,0.004612,10,0.001476


In [17]:
plot_var_vs_loss(train_var, residualbank, calendar=calendar_last_180_days)

In [18]:
# train + forecast birlikte (bank'in tum gunleri); ilk multi_day_var_min_periods gun yine dusulur, run kaydedilmez
entire_var = calendar_last_180_days.var_from_residual_bank_entire_period(
    residualbank,
    var_confidence=VAR_CONFIDENCE,
    multi_day_var_calculation_window_size=60,
    multi_day_var_min_periods=30,
)
plot_var_vs_loss(entire_var, residualbank, title="VaR vs Realized Loss (train + forecast)", calendar=calendar_last_180_days)

In [19]:
# ayni calendar, uc donem yan yana (her biri kendi basina tek pencere). var_confidence: VaR'i uretirken kullandigimizla ayni olmali
backtest_long = dm.backtest_periods_side_by_side(
    calendar_last_180_days,
    var_confidence=VAR_CONFIDENCE,
    forecast_var_df=var_df,      # out-of-sample (durust test)
    train_var_df=train_var,      # in-sample: iyimser, saglik kontrolu
    entire_var_df=entire_var,    # train + forecast tek pencere
)
backtest_long

/Users/kaan/Developer/MLProject-standalone/core/var_backtest_utils.py:459: UserWarning: backtest_train_period: var_df covers 115 of the 126 train-period calendar dates; the backtest runs on those 115.
  columns.append(backtest_train_period(calendar, train_var_df, var_confidence, test_confidence))
/Users/kaan/Developer/MLProject-standalone/core/var_backtest_utils.py:461: UserWarning: backtest_entire_period: var_df covers 116 of the 147 entire-period calendar dates; the backtest runs on those 116.
  columns.append(backtest_entire_period(calendar, entire_var_df, var_confidence, test_confidence))


,forecast,train,entire
var_confidence,0.95,0.95,0.95
test_confidence,0.95,0.95,0.95
n_days,21,115,116
n_exceedances,2,13,10
exceedance_rate,0.095238,0.113043,0.086207
expected_exceedance_rate,0.05,0.05,0.05
binomial_pass,True,False,True
traffic_light_zone,1,2,2
kupiec_pof_stat,0.723402,7.201619,2.656689
kupiec_pof_critical_value,3.841459,3.841459,3.841459


## Wide format

Tek bir model, doğrudan **portföy** varyansını tahmin eder (ticker bazlı tahmin ve EWMA bank yok). Aynı calendar kullanılır, bank'lar ve değişkenler long bölümünden ayrı adlandırılmıştır (`*_wide`).

In [20]:
training_feature_wide, forecast_feature_wide = fb.build_wide_features(calendar_last_180_days, target_scale=10000)

[INFO] wide train: 126 bars -> 126 after dropna (0 dropped, 0.0%)
[INFO] wide forecast: 22 bars -> 22 after dropna (0 dropped, 0.0%)


In [21]:
# residual bank da tek kullanimlik: her egitim icin yeni bir tane (wide yolda EwmaVarianceBank yok)
residual_bank_wide = dm.ResidualBank(
    mu_mode="constant",
    default_mu_hat=0.0,
    rolling_mu_hat_window_size=50,              # only read when mu_mode="rolling"
)

model_wide = tm.xgboost_train(calendar_last_180_days, training_feature_wide, residual_bank=residual_bank_wide)

In [22]:
# forecast penceresini model bank'inin kopyasina yazar (VaR yok), sonra VaR bank'tan okunur
residualbank_wide = model_wide.calculate_multi_day_standart_residuals_from_wide_format_model(
    calendar=calendar_last_180_days,
    forecast_feature=forecast_feature_wide,
)

var_df_wide = calendar_last_180_days.var_from_residual_bank_forecast_period(
    residualbank_wide,
    var_confidence=VAR_CONFIDENCE,
    multi_day_var_calculation_window_size=360,      # int: use at most the last N residuals; None: use all residuals available
    run_key="xgboost_wide",
    multi_day_var_min_periods=30,
)
var_df_wide.head()

,varForecast,n_residuals_used,loss
Date,,,
2024-12-02,0.008277,125,0.003004
2024-12-03,0.010309,126,-0.000849
2024-12-04,0.010106,127,0.013437
2024-12-05,0.006107,128,-0.006114
2024-12-06,0.010966,129,0.002476


In [23]:
plot_var_vs_loss(var_df_wide, residualbank_wide, calendar=calendar_last_180_days)

In [24]:
# wide modelin forecast donemi backtest'i (tekli fonksiyon: Series doner, adi "forecast")
dm.backtest_forecast_period(calendar_last_180_days, var_df_wide, VAR_CONFIDENCE)

var_confidence                           0.95
test_confidence                          0.95
n_days                                     21
n_exceedances                               2
exceedance_rate                      0.095238
expected_exceedance_rate                 0.05
binomial_pass                            True
traffic_light_zone                          1
kupiec_pof_stat                      0.723402
kupiec_pof_critical_value            3.841459
kupiec_pof_pass                          True
kupiec_tuff_pass                         True
haas_tbf_pass                            True
christoffersen_ind_stat              0.445363
christoffersen_ind_critical_value    3.841459
christoffersen_ind_pass                  True
christoffersen_cc_stat               1.168765
christoffersen_cc_critical_value     5.991465
christoffersen_cc_pass                   True
Name: forecast, dtype: object

## Modelleri karşılaştırma (aynı calendar)

Long ve wide, aynı `calendar_last_180_days` üzerinde çalıştı. Aşağıda VaR'ları ve model çıktılarını (predicted_variance) tek grafikte, gerçekleşen EWMA varyansıyla birlikte görüyoruz.

In [25]:
from graph_helpers import plot_var_models_vs_loss, plot_models_output_vs_realized_variance, plot_model_output_vs_realized_variance

# {model_adi: (var_df, residualbank)}, hepsi ayni calendar'dan olmali (loss kolonlari tutarsizsa hata verir)
plot_var_models_vs_loss({
    "xgboost_long": (var_df, residualbank),
    "xgboost_wide": (var_df_wide, residualbank_wide),
}, calendar=calendar_last_180_days)

In [26]:
# gerceklesen getirilerle doldurulan EWMA bank (model karisik degil): warmup ile isitilir, sonra train + forecast eklenir.
# bank'in d gunu degeri d-1'e kadar olan getirilerden hesaplanir (d'nin kendi getirisi dahil degil)
realized_bank = calendar_last_180_days.ewma_realized_variance_bank(ewma_lam=0.94, ewma_min_periods=30)

# gercek realized variance: d gununun degeri d'nin kendi getirisini (r_d) icerir = bank'in bir sonraki bar degeri
realized_variance = calendar_last_180_days.ewma_realized_variance(ewma_lam=0.94, ewma_min_periods=30)
realized_variance.head()

Date
2024-06-03    0.000061
2024-06-04    0.000058
2024-06-05    0.000054
2024-06-06    0.000051
2024-06-07    0.000049
Name: ewma_realized_variance, dtype: float64

In [27]:
# tek model: model_output vs gerceklesen EWMA varyans (sadece forecast penceresi)
plot_model_output_vs_realized_variance(calendar_last_180_days, residualbank, model_name="xgboost_long")

In [28]:
# birden fazla model: ayni calendar, tum bank tarihleri (egitim in-sample + forecast)
plot_models_output_vs_realized_variance(
    calendar_last_180_days,
    {"xgboost_long": residualbank, "xgboost_wide": residualbank_wide},
    only_forecast_window=False,     # True: sadece forecast penceresi
)